# Data leakage deep-dive
## Who is really in your test set?

A random split *feels* rigorous. But if a test molecule is identical (or nearly identical) to a training molecule, your metrics measure **memorization**, not generalization. This is the single most common way molecular-ML results get inflated.

chemcheck distinguishes **three flavors** of cross-split contamination. You will plant one of each, see how they differ, and fix them:

In [1]:
import pandas as pd

train = [
    ("t01", "CC(C)Cc1ccc(C(C)C(=O)O)cc1", 5.2),  # ibuprofen
    ("t02", "CCO",                         1.0),  # ethanol
    ("t03", "CC(=O)c1ccccc1",               4.2),  # acetophenone
    ("t04", "C1CCCCC1N",                    2.2),  # cyclohexylamine
    ("t05", "c1ccc2ncncc2c1",               3.3),  # quinazoline
]
test = [
    # id,    smiles,                              activity   # planted contamination
    ("e01", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",      5.1),      # FLAVOR 1: byte-identical to t01
    ("e02", "CCC(C)Cc1ccc(C(C)C(=O)O)cc1",     5.4),      # FLAVOR 2: ibuprofen + one extra carbon
    ("e03", "c1ccccc1C(=O)O",                  3.9),      # FLAVOR 3: new molecule, familiar scaffold
    ("e04", "C1CCOC1",                         2.0),      # clean: unseen scaffold, dissimilar
]
cols = ["id", "smiles", "activity"]
pd.DataFrame(train, columns=cols).to_csv("train.csv", index=False)
pd.DataFrame(test, columns=cols).to_csv("test.csv", index=False)
print("train:", len(train), " test:", len(test))

train: 5  test: 4


## Audit the split

Two files = train/test mode. No `--split-col` needed:

In [2]:
from chemcheck.report import audit, render_terminal

rep0 = audit(["train.csv", "test.csv"], label_col="activity")
print(render_terminal(rep0))


✓ 9 valid molecules / ✕ 0 invalid (n=9)
✕ [error] near-duplicates across splits (Tc≥0.95): 2 (22.2%)  [cross_split_near_duplicates]
    e.g. {"train_row": "t01", "test_row": "e01", "tanimoto": 1.0}
    → Near-duplicates must live in one split only — move the test copy to train or drop it; otherwise metrics measure memorization.
    (1 cross-split pairs above threshold.)
✕ [error] 2D-identity leakage (1 test molecules seen in train): 1 (11.1%)  [identity_leakage]
    e.g. {"train_rows": ["t01"], "test_row": "e01", "connectivity": "CC(C)Cc1ccc(C(C)C(=O)O)cc1"}
    → Remove leaked test molecules or move them to train; identical structures must never span splits.
    (Stereo-agnostic match (connectivity SMILES) across train/test.)
⚠ [warning] molecules with unspecified stereocenters: 3 (33.3%)  [unspecified_stereo]
    e.g. {"row": "t01", "smiles": "CC(C)Cc1ccc(C(C)C(=O)O)cc1", "unassigned_centers": 1}
    e.g. {"row": "e01", "smiles": "CC(C)Cc1ccc(C(C)C(=O)O)cc1", "unassigned_centers": 1

## The three flavors, side by side

| test mol | max Tc to train | scaffold seen in train? | finding | verdict |
|---|---|---|---|---|
| e01 (identical) | 1.00 | yes | `identity_leakage` (**error**) | must fix (pure memorization) |
| e02 (analog) | 0.71 | yes | `analog_leakage` (**warning**) | probably fix (same chemical series) |
| e03 (benzoic acid) | 0.58 | yes | `scaffold_overlap` only | judgment call (same scaffold, different chemistry) |
| e04 (THF) | 0.12 | no | — | genuinely novel |

Note the thresholds: identity is stereo-agnostic exact match; analogs are Morgan Tc ≥ 0.6 (tunable with `--analog-thresh`); scaffold overlap uses Bemis–Murcko scaffolds. Let's look at the actual evidence objects:

In [3]:
for cid in ("identity_leakage", "analog_leakage", "scaffold_overlap"):
    f = next((x for x in rep0.findings if x.check_id == cid), None)
    print(f"=== {cid}: {f.title} [{f.severity.value}] ===")
    for ex in f.examples:
        interesting = {k: (str(v)[:60]) for k, v in ex.items()
                       if k in ("train_row", "test_row", "tanimoto", "test_row",
                                "shared_scaffold", "scaffold", "recommendation")}
        print("   ", interesting)
    print("   →", f.recommendation[:160], "\n")

=== identity_leakage: 2D-identity leakage (1 test molecules seen in train) [error] ===
    {'test_row': 'e01'}
   → Remove leaked test molecules or move them to train; identical structures must never span splits. 

=== analog_leakage: analog leakage: 2 test molecules with Tc≥0.6 to train [warning] ===
    {'train_row': 't01', 'test_row': 'e01', 'tanimoto': '1.0', 'shared_scaffold': 'c1ccccc1', 'recommendation': 'move test molecule to train OR use scaffold/cluster split'}
    {'train_row': 't01', 'test_row': 'e02', 'tanimoto': '0.71', 'shared_scaffold': 'c1ccccc1', 'recommendation': 'move test molecule to train OR use scaffold/cluster split'}
   → Inspect the worst pairs below; regroup analog series into the same split. Median max test→train similarity is 0.64 — above ~0.6 the split is easier than it look 

=== scaffold_overlap: scaffold split overlap (75.0% of test scaffolds seen in train) [warning] ===
    {'test_row': 'e01', 'scaffold': 'c1ccccc1'}
    {'test_row': 'e02', 'scaffold':

## Fix round 1: remove the identical molecule

e01 contributes nothing a model couldn't answer by lookup. Drop it from test (or move it to train):

In [4]:
tr = pd.read_csv("train.csv")
te = pd.read_csv("test.csv")
te1 = te[te["id"] != "e01"].copy()          # e01 removed from test
te1.to_csv("test_fix1.csv", index=False)

rep1 = audit(["train.csv", "test_fix1.csv"], label_col="activity")
ids1 = {f.check_id for f in rep1.findings}
print(f"score: {rep0.score} → {rep1.score}")
print("identity_leakage gone:", "identity_leakage" not in ids1)
print("analog_leakage still there:", "analog_leakage" in ids1, "(e02 is still a near-neighbor of t01)")

score: 71 → 82
identity_leakage gone: True
analog_leakage still there: True (e02 is still a near-neighbor of t01)


## Fix round 2: regroup the analog series

The standard remedy is to keep a chemical series in **one** split. Move e02 to train so the test set only contains what the model hasn't effectively seen:

In [5]:
tr2 = pd.concat([tr, te[te["id"] == "e02"]], ignore_index=True)
te2 = te[~te["id"].isin(["e01", "e02"])].copy()
tr2.to_csv("train_fix2.csv", index=False)
te2.to_csv("test_fix2.csv", index=False)

rep2 = audit(["train_fix2.csv", "test_fix2.csv"], label_col="activity")
ids2 = {f.check_id for f in rep2.findings}
print(f"score: {rep0.score} → {rep1.score} → {rep2.score}")
print("identity_leakage gone:", "identity_leakage" not in ids2)
print("analog_leakage gone:", "analog_leakage" not in ids2)

score: 71 → 82 → 93
identity_leakage gone: True
analog_leakage gone: True


## What about e03 (scaffold overlap)?

e03 shares the benzene scaffold with train but is chemically distinct (Tc 0.58). chemcheck flags it as *information*, not an error, and that's deliberate because:

- For a **lead-optimization** model ("rank my next analog"), e03-like test points are fine, even desirable.
- For a **generalization** claim ("works on new chemotypes"), you want a scaffold or cluster split with zero overlap.

The finding tells you which regime you're in. If you need the strict regime, re-split by scaffold so the whole benzene series lands on one side. Then `scaffold_overlap` will disappear too.

## Takeaways

1. **Identity leakage is never acceptable.** Same connectivity in train and test = inflated metrics, always.
2. **Analog leakage is the silent one.** Tc ≥ 0.6 test molecules let models interpolate instead of generalize.
3. **Scaffold overlap is context-dependent.** Report it, then match the split to the claim.
4. Re-audit after every fix. The score should only go in one direction.

In [6]:
ids0 = {f.check_id for f in rep0.findings}
assert {"identity_leakage", "analog_leakage", "scaffold_overlap"} <= ids0
assert "identity_leakage" not in ids2 and "analog_leakage" not in ids2
assert rep2.score > rep0.score
print(f"leakage drills passed ({rep0.score} → {rep2.score})")

leakage drills passed (71 → 93)
